# Bronze — trust price history

`landing.trust_prices_raw` → `bronze.trust_prices`. Every column cast to STRING.

The known problems **must survive this layer**: the mixed date labels, the ~25
tickers on a wrong scale, and the `PCFT` zero price. Silver fixes them, and the fix
is only auditable if Bronze still holds the original.

Expected: **16,357 rows**, same as Landing.

In [0]:
%sql
-- price stays STRING: casting to DOUBLE here would turn any malformed value into a
-- silent NULL, which is exactly the rejection this layer is not allowed to do.
CREATE OR REPLACE TABLE `index-vs-trust-pipeline`.bronze.trust_prices AS
SELECT
  CAST(trust_name        AS STRING) AS trust_name,
  CAST(ticker            AS STRING) AS ticker,
  CAST(aic_sector        AS STRING) AS aic_sector,
  CAST(`date`            AS STRING) AS `date`,
  CAST(price_gbx_or_gbp  AS STRING) AS price_gbx_or_gbp
FROM `index-vs-trust-pipeline`.landing.trust_prices_raw;

## Verification

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.trust_prices_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trust_prices)      AS bronze_rows,
  (SELECT COUNT(DISTINCT ticker) FROM `index-vs-trust-pipeline`.bronze.trust_prices) AS tickers;

Expect **16,357 / 16,357 / 102**.

In [0]:
%sql
-- The known-bad row proves Bronze cleaned nothing. Must still read 0.000.
SELECT ticker, `date`, price_gbx_or_gbp
FROM `index-vs-trust-pipeline`.bronze.trust_prices
WHERE ticker = 'PCFT' AND `date` = '2019-11-01';

In [0]:
%sql
DESCRIBE TABLE `index-vs-trust-pipeline`.bronze.trust_prices;